In [1]:
#import
import pandas as pd
import rdata
import numpy as np
import tensorflow as tf
import os

from pathlib import Path
from urllib.request import urlretrieve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from keras.callbacks import TensorBoard, ModelCheckpoint


import datetime


In [25]:

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

# URL base del repositorio oficial donde están los archivos .rda
BASE_URL = "https://github.com/paezha/idealista18/raw/master/data"

# Carpeta local donde guardaremos los archivos descargados
DATA_DIR = Path("idealista18_data")
DATA_DIR.mkdir(exist_ok=True)

# Relación entre ciudad y archivo de ventas correspondiente
SALE_FILES = {
    "Barcelona": "Barcelona_Sale.rda",
    "Madrid": "Madrid_Sale.rda",
    "Valencia": "Valencia_Sale.rda",
}


# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def download_file(filename: str) -> Path:
    """
    Descarga un archivo desde el repositorio oficial si no existe en local.

    Parameters
    ----------
    filename : str
        Nombre del archivo a descargar, por ejemplo 'Madrid_Sale.rda'.

    Returns
    -------
    Path
        Ruta local al archivo descargado.
    """
    local_path = DATA_DIR / filename

    if not local_path.exists():
        url = f"{BASE_URL}/{filename}"
        print(f"Descargando {filename}...")
        urlretrieve(url, local_path)

    return local_path


def load_rda(filename: str) -> dict:
    """
    Carga un archivo .rda y devuelve su contenido como diccionario.

    Nota:
    Un archivo .rda puede contener uno o varios objetos de R.
    Esta librería normalmente los devuelve en un diccionario
    donde la clave es el nombre del objeto almacenado.

    Parameters
    ----------
    filename : str
        Nombre del archivo .rda.

    Returns
    -------
    dict
        Diccionario con los objetos cargados desde el archivo.
    """
    local_path = download_file(filename)
    data = rdata.read_rda(str(local_path), default_encoding="utf8")
    return data


def load_city_sales(city_name: str, filename: str) -> pd.DataFrame:
    """
    Carga el dataset de ventas de una ciudad y añade una columna CITY.

    Parameters
    ----------
    city_name : str
        Nombre de la ciudad, por ejemplo 'Madrid'.
    filename : str
        Nombre del archivo .rda asociado.

    Returns
    -------
    pd.DataFrame
        DataFrame con los datos de ventas de esa ciudad.
    """
    rda_content = load_rda(filename)

    # El objeto dentro del .rda suele llamarse igual que el archivo sin extensión
    object_name = filename.replace(".rda", "")

    if object_name not in rda_content:
        raise KeyError(
            f"No se encontró el objeto '{object_name}' dentro de '{filename}'. "
            f"Objetos disponibles: {list(rda_content.keys())}"
        )

    df = rda_content[object_name].copy()

    # Añadimos una columna para que el modelo sepa de qué ciudad viene cada vivienda
    df["CITY"] = city_name

    return df


def load_all_sales() -> pd.DataFrame:
    """
    Carga y concatena los datasets de ventas de Barcelona, Madrid y Valencia.

    Returns
    -------
    pd.DataFrame
        DataFrame unificado con las 3 ciudades.
    """
    city_dfs = []

    for city_name, filename in SALE_FILES.items():
        print(f"Cargando datos de {city_name}...")
        city_df = load_city_sales(city_name, filename)
        city_dfs.append(city_df)

    df_all = pd.concat(city_dfs, ignore_index=True)
    return df_all


# ============================================================
# CARGA PRINCIPAL
# ============================================================

# Cargar las tres ciudades en un único DataFrame
df = load_all_sales()


# ============================================================
# INSPECCIÓN BÁSICA
# ============================================================

print("\nPrimeras filas:")
print(df.head())

print("\nDimensiones del dataset unificado:")
print(df.shape)

print("\nCiudades incluidas:")
print(df["CITY"].value_counts())

print("\nColumnas disponibles:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos por columna:")
print(df.isnull().sum().sort_values(ascending=False).head(20))

Cargando datos de Barcelona...


C:\Users\emartincar2\AppData\Roaming\Python\Python311\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "bbox". The underlying R object is returned instead.
  warnings.warn(
C:\Users\emartincar2\AppData\Roaming\Python\Python311\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "crs". The underlying R object is returned instead.
  warnings.warn(
C:\Users\emartincar2\AppData\Roaming\Python\Python311\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "XY". The constructor for class "POINT" will be used instead.
  warnings.warn(
C:\Users\emartincar2\AppData\Roaming\Python\Python311\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "POINT". The constructor for class "sfg" will be used instead.
  warnings.warn(
C:\Users\emartincar2\AppData\Roaming\Python\Python311\site-packages\rdata\conversion\_conversion.py:90

Cargando datos de Madrid...


C:\Users\emartincar2\AppData\Roaming\Python\Python311\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "bbox". The underlying R object is returned instead.
  warnings.warn(
C:\Users\emartincar2\AppData\Roaming\Python\Python311\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "crs". The underlying R object is returned instead.
  warnings.warn(
C:\Users\emartincar2\AppData\Roaming\Python\Python311\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "XY". The constructor for class "POINT" will be used instead.
  warnings.warn(
C:\Users\emartincar2\AppData\Roaming\Python\Python311\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "POINT". The constructor for class "sfg" will be used instead.
  warnings.warn(
C:\Users\emartincar2\AppData\Roaming\Python\Python311\site-packages\rdata\conversion\_conversion.py:90

Cargando datos de Valencia...


C:\Users\emartincar2\AppData\Roaming\Python\Python311\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "bbox". The underlying R object is returned instead.
  warnings.warn(
C:\Users\emartincar2\AppData\Roaming\Python\Python311\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "crs". The underlying R object is returned instead.
  warnings.warn(
C:\Users\emartincar2\AppData\Roaming\Python\Python311\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "XY". The constructor for class "POINT" will be used instead.
  warnings.warn(
C:\Users\emartincar2\AppData\Roaming\Python\Python311\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "POINT". The constructor for class "sfg" will be used instead.
  warnings.warn(
C:\Users\emartincar2\AppData\Roaming\Python\Python311\site-packages\rdata\conversion\_conversion.py:90


Primeras filas:
                 ASSETID  PERIOD     PRICE    UNITPRICE  CONSTRUCTEDAREA  \
0  A11898131848556022319  201803  323000.0  3845.238095               84   
1  A18099432772155664747  201803  217000.0  2583.333333               84   
2   A2003099089407882787  201803  114000.0  1407.407407               81   
3   A1010373782315301134  201803  378000.0  4784.810127               79   
4  A12978912200216838006  201803  434000.0  3909.909910              111   

   ROOMNUMBER  BATHNUMBER  HASTERRACE  HASLIFT  HASAIRCONDITIONING  ...  \
0           4           1           1        1                   1  ...   
1           3           2           0        1                   1  ...   
2           2           1           0        1                   1  ...   
3           2           1           0        1                   0  ...   
4           4           2           1        1                   1  ...   

   BUILTTYPEID_3  DISTANCE_TO_CITY_CENTER  DISTANCE_TO_METRO  \
0          

In [26]:
df["CITY"].value_counts()


CITY
Madrid       94815
Barcelona    61486
Valencia     33622
Name: count, dtype: int64

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 189923 entries, 0 to 189922
Data columns (total 45 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   ASSETID                        189923 non-null  object 
 1   PERIOD                         189923 non-null  Int32  
 2   PRICE                          189923 non-null  float64
 3   UNITPRICE                      189923 non-null  float64
 4   CONSTRUCTEDAREA                189923 non-null  Int32  
 5   ROOMNUMBER                     189923 non-null  Int32  
 6   BATHNUMBER                     189923 non-null  Int32  
 7   HASTERRACE                     189923 non-null  Int32  
 8   HASLIFT                        189923 non-null  Int32  
 9   HASAIRCONDITIONING             189923 non-null  Int32  
 10  AMENITYID                      189923 non-null  Int32  
 11  HASPARKINGSPACE                189923 non-null  Int32  
 12  ISPARKINGSPACEINCLUDEDINPRICE 

In [28]:

# DATOS

target = "PRICE"

features = [
    "CONSTRUCTEDAREA",
    "ROOMNUMBER",
    "BATHNUMBER",
    "HASTERRACE",
    "HASLIFT",
    "HASAIRCONDITIONING",
    "HASPARKINGSPACE",
    "HASBOXROOM",
    "HASWARDROBE",
    "HASSWIMMINGPOOL",
    "HASGARDEN",
    "ISDUPLEX",
    "ISSTUDIO",
    "ISINTOPFLOOR",
    "LATITUDE",
    "LONGITUDE",
    "DISTANCE_TO_CITY_CENTER",
    "DISTANCE_TO_METRO",
    "CITY",
]

df_model = df[features + [target]].copy()
df_model = df_model.dropna()

# Convertir CITY a variables dummy
df_model = pd.get_dummies(df_model, columns=["CITY"], drop_first=True)
df_model.columns = [str(col) for col in df_model.columns]

# Separar X e y
X = df_model.drop(columns=[target]).copy()
y = df_model[target].copy()

# Train / temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# Validation / test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

#Escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

Train: (132946, 20) (132946,)
Val: (28488, 20) (28488,)
Test: (28489, 20) (28489,)


In [29]:
def crear_modelo(dropout_rate, input_dim):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(dropout_rate),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(dropout_rate),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(1)
    ])
    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )
    return model

In [30]:
model = crear_modelo(dropout_rate=0.2, input_dim=X_train_scaled.shape[1])
model.summary()



Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 128)            │         2,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,057 (51.00 KB)

 Trainable params: 13,057 (51.00 KB)

 Non-trainable params: 0 (0.00 B)

In [31]:
#FUNCION PARA CALLBACKS
def crear_callbacks(dropout_rate):
    """
    Crea la lista de callbacks para un experimento con un valor
    concreto de dropout.

    Parameters
    ----------
    dropout_rate : float
        Valor de dropout del experimento.

    Returns
    -------
    list
        Lista de callbacks para usar en model.fit()
    """

    # Carpeta donde TensorBoard guardará los logs del entrenamiento
    log_dir = os.path.join(
        "logs",
        f"dropout_{dropout_rate}",
        datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    )

    # Ruta donde se guardará el mejor modelo de este experimento
    checkpoint_path = os.path.join(
        "checkpoints",
        f"mejor_modelo_dropout_{dropout_rate}.keras"
    )

    # Crear carpeta de checkpoints si no existe
    os.makedirs("checkpoints", exist_ok=True)

    callbacks = [
        # Guarda automáticamente el mejor modelo según val_mae
        tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_mae",
            save_best_only=True,
            mode="min",
            verbose=1
        ),

        # Guarda logs para TensorBoard
        tf.keras.callbacks.TensorBoard(
            log_dir=log_dir,
            histogram_freq=1
        ),

        # Detiene el entrenamiento si la validación no mejora
        tf.keras.callbacks.EarlyStopping(
            monitor="val_mae",
            patience=10,
            restore_best_weights=True,
            mode="min",
            verbose=1
        )
    ]

    return callbacks

In [35]:
#Entrenar un modelo
model = crear_modelo(dropout_rate=0.2, input_dim=X_train_scaled.shape[1])

callbacks = crear_callbacks(dropout_rate=0.2)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100
4125/4155 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 114508477912.4053 - mae: 199658.0214
Epoch 1: val_mae improved from None to 88506.07812, saving model to checkpoints\mejor_modelo_dropout_0.2.keras

Epoch 1: finished saving model to checkpoints\mejor_modelo_dropout_0.2.keras
4155/4155 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 56047763456.0000 - mae: 132736.2969 - val_loss: 24850378752.0000 - val_mae: 88506.0781
Epoch 2/100
4142/4155 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 25921047010.0859 - mae: 91699.2641
Epoch 2: val_mae improved from 88506.07812 to 82149.79688, saving model to checkpoints\mejor_modelo_dropout_0.2.keras

Epoch 2: finished saving model to checkpoints\mejor_modelo_dropout_0.2.keras
4155/4155 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - loss: 24732176384.0000 - mae: 90190.3203 - val_loss: 21598814208.0000 - val_mae: 82149.7969
Epoch 3/100
4136/4155 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 23018935378.9400 - mae: 87472.5607
Epoch 3: val_mae improved from 82149.79688 t

In [37]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Valores de dropout que quieres comparar
dropouts = [0.2, 0.5, 0.8]

# Aquí guardaremos los resultados de cada experimento
resultados = []

# Aquí guardaremos también el history de cada modelo por si luego quieres revisarlo
histories = {}

for dr in dropouts:
    print(f"\n{'='*60}")
    print(f"ENTRENANDO MODELO CON DROPOUT = {dr}")
    print(f"{'='*60}")

    # 1. Crear un modelo nuevo para este valor de dropout
    model = crear_modelo(
        dropout_rate=dr,
        input_dim=X_train_scaled.shape[1]
    )

    # 2. Crear callbacks específicos para este experimento
    callbacks = crear_callbacks(dropout_rate=dr)

    # 3. Entrenar el modelo
    history = model.fit(
        X_train_scaled,
        y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=100,
        batch_size=32,
        callbacks=callbacks,
        verbose=1
    )

    # Guardar history por si luego quieres inspeccionarlo
    histories[dr] = history.history

    # 4. Hacer predicciones sobre validation
    y_val_pred = model.predict(X_val_scaled).flatten()

    # 5. Calcular métricas de validación
    val_mae = mean_absolute_error(y_val, y_val_pred)
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    val_r2 = r2_score(y_val, y_val_pred)

    # 6. Guardar resultados
    resultados.append({
        "dropout": dr,
        "val_mae": val_mae,
        "val_rmse": val_rmse,
        "val_r2": val_r2,
        "epochs_entrenadas": len(history.history["loss"])
    })

# Convertir resultados a DataFrame para compararlos mejor
df_resultados = pd.DataFrame(resultados).sort_values(by="val_mae")

print("\nRESULTADOS FINALES DE VALIDACIÓN")
print(df_resultados)


ENTRENANDO MODELO CON DROPOUT = 0.2
Epoch 1/100
4138/4155 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 105921585677.6104 - mae: 192478.2701
Epoch 1: val_mae improved from None to 86724.62500, saving model to checkpoints\mejor_modelo_dropout_0.2.keras

Epoch 1: finished saving model to checkpoints\mejor_modelo_dropout_0.2.keras
4155/4155 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 52266024960.0000 - mae: 128359.3984 - val_loss: 23444434944.0000 - val_mae: 86724.6250
Epoch 2/100
4128/4155 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 24354255234.2326 - mae: 89896.7436
Epoch 2: val_mae improved from 86724.62500 to 82203.50000, saving model to checkpoints\mejor_modelo_dropout_0.2.keras

Epoch 2: finished saving model to checkpoints\mejor_modelo_dropout_0.2.keras
4155/4155 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 23976992768.0000 - mae: 88741.0000 - val_loss: 21410537472.0000 - val_mae: 82203.5000
Epoch 3/100
4126/4155 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 23457063440.4421 - mae: 86945.8279
Epoch 3

In [ ]:
print(df_resultados)

   dropout       val_mae       val_rmse    val_r2  epochs_entrenadas
0      0.2  72188.077766  134030.925440  0.854998                100
1      0.5  74461.117031  137287.716744  0.847866                 64
2      0.8  84422.322425  155316.063251  0.805287                 19


In [41]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mejor_modelo = tf.keras.models.load_model("checkpoints/mejor_modelo_dropout_0.2.keras")

y_test_pred = mejor_modelo.predict(X_test_scaled).flatten()

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

print("=== TEST ===")
print(f"MAE:  {test_mae:,.2f}")
print(f"RMSE: {test_rmse:,.2f}")
print(f"R²:   {test_r2:.4f}")

891/891 ━━━━━━━━━━━━━━━━━━━━ 1s 616us/step
=== TEST ===
MAE:  73,877.71
RMSE: 135,367.34
R²:   0.8561
